# Тренування моделі оцінки кредитного ризику (Credit Scoring Model)
**Версія:** 3.1.0 (Оновлена)
Цей блокнот виконує завантаження даних LendingClub, їх очищення, генерацію нових ознак (Feature Engineering) та тренування моделі `RandomForest` з пошуком оптимальних гіперпараметрів (`GridSearchCV`).

In [20]:
# --- МОДУЛЬ 1: ІМПОРТ БІБЛІОТЕК ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib
import warnings

warnings.filterwarnings('ignore')
print("Бібліотеки успішно завантажені.")

Бібліотеки успішно завантажені.


In [21]:
# --- МОДУЛЬ 2: ЗАВАНТАЖЕННЯ ДАНИХ ---
# Вкажи правильний шлях до свого розпакованого CSV файлу
# Додаємо літеру 'r' перед рядком, щоб Windows правильно читав зворотні слеші
file_path = r'D:\GitFile\engineer_work\data\accepted_2007_to_2018Q4.csv' 
df = pd.read_csv(file_path, nrows=200000)

# Завантажуємо лише частину даних для швидкості (або прибери nrows, щоб завантажити все)
print("Завантаження даних...")
df = pd.read_csv(file_path, nrows=200000) 
print(f"Завантажено рядків: {df.shape[0]}, колонок: {df.shape[1]}")

Завантаження даних...
Завантажено рядків: 200000, колонок: 151


In [22]:
# --- МОДУЛЬ 3: ОЧИЩЕННЯ ТА ФІЛЬТРАЦІЯ (DATA CLEANING) ---

# 1. Залишаємо лише завершені кредити (уникаємо витоку даних з поточних)
valid_statuses = ['Fully Paid', 'Charged Off']
df = df[df['loan_status'].isin(valid_statuses)].copy()

# 2. Створюємо цільову змінну (1 - Успіх, 0 - Дефолт)
df['target'] = df['loan_status'].apply(lambda x: 1 if x == 'Fully Paid' else 0)

# 3. Фільтруємо аномалії (Outliers)
df = df[(df['annual_inc'] >= 1000) & (df['annual_inc'] <= 300000)]
df = df[df['dti'] <= 100]

# 4. Видаляємо пропущені значення у критичних колонках
cols_to_check = ['emp_length', 'home_ownership', 'purpose', 'fico_range_low']
df = df.dropna(subset=cols_to_check)

print(f"Залишилось записів після очищення: {df.shape[0]}")
print("Баланс класів:\n", df['target'].value_counts(normalize=True))

Залишилось записів після очищення: 163874
Баланс класів:
 target
1    0.803556
0    0.196444
Name: proportion, dtype: float64


In [23]:
# --- МОДУЛЬ 4: ІНЖЕНЕРІЯ ОЗНАК (FEATURE ENGINEERING) ---

# 1. Перетворюємо 'term' (напр. " 36 months") у число 36
df['term'] = df['term'].astype(str).str.extract('(\d+)').astype(float)

# 2. Створюємо потужну бізнес-метрику: відношення суми кредиту до доходу
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + 1)

print("Інженерія ознак завершена.")

Інженерія ознак завершена.


In [24]:
# --- МОДУЛЬ 5: ПІДГОТОВКА ДАНИХ ДО ML (ENCODING & SPLITTING) ---

# Визначаємо фінальний список колонок, які доступні банку ДО видачі кредиту
features = [
    'annual_inc', 'loan_amnt', 'term', 'fico_range_low', 'dti', 'loan_to_income', 
    'emp_length', 'home_ownership', 'purpose'
]

X = df[features].copy()
y = df['target']

# Перетворюємо текстові колонки у формат One-Hot Encoding (0 та 1)
# drop_first=True зменшує розмірність і уникає мультиколінеарності
X = pd.get_dummies(X, columns=['emp_length', 'home_ownership', 'purpose'], drop_first=True)

# Зберігаємо список фінальних колонок для додатку app.py
final_columns = X.columns.tolist()
joblib.dump(final_columns, 'model_columns.pkl')
print(f"Кількість фічей після One-Hot Encoding: {len(final_columns)}")

# Розбиваємо на тренувальну та тестову вибірки (Stratify зберігає пропорції класів)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Розбиття завершено: X_train shape:", X_train.shape)

Кількість фічей після One-Hot Encoding: 31
Розбиття завершено: X_train shape: (131099, 31)


In [25]:
# --- МОДУЛЬ 6: ТРЕНУВАННЯ МОДЕЛІ ТА ТЮНІНГ (GRID SEARCH) ---

# Базова модель (балансуємо класи через class_weight)
rf_base = RandomForestClassifier(class_weight='balanced', random_state=42)

# Гіперпараметри для перевірки (можеш розширити список, але це збільшить час навчання)
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [10, 15],
    'min_samples_split': [5, 10]
}

# Налаштовуємо крос-валідацію (3 фолди)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Ініціалізуємо пошук
grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    cv=cv, 
    scoring='roc_auc', 
    n_jobs=-1, # Використовувати всі ядра процесора
    verbose=2
)

print("Починаємо навчання. Зачекайте...")
grid_search.fit(X_train, y_train)

# Зберігаємо найкращу модель
best_rf_model = grid_search.best_estimator_
print(f"\nНайкращі знайдені параметри: {grid_search.best_params_}")

Починаємо навчання. Зачекайте...
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Найкращі знайдені параметри: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}


In [26]:
# --- МОДУЛЬ 7: ОЦІНКА ТА ЗБЕРЕЖЕННЯ ---

# Робимо прогнози
y_pred = best_rf_model.predict(X_test)
y_pred_proba = best_rf_model.predict_proba(X_test)[:, 1]

# Виводимо метрики
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Зберігаємо саму модель для Flask-додатку
joblib.dump(best_rf_model, 'credit_model.pkl')
print("Модель успішно збережена у файл 'credit_model.pkl'!")

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.34      0.57      0.42      6438
           1       0.87      0.73      0.79     26337

    accuracy                           0.70     32775
   macro avg       0.61      0.65      0.61     32775
weighted avg       0.77      0.70      0.72     32775

ROC-AUC Score: 0.7058
Модель успішно збережена у файл 'credit_model.pkl'!
